In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score

train = pd.read_csv("train_features.csv")
val = pd.read_csv("val_features.csv")

In [2]:
features = [
    "num_items", "total_freight", "log_total_price",
    "num_payments", "total_payment_value",
    "purchase_month", "purchase_dayofweek",
    "same_state", "num_unique_sellers"
]

X_train, y_train = train[features], train["is_late"]
X_val, y_val = val[features], val["is_late"]

In [3]:
# Baseline
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
print("Baseline AUC:", roc_auc_score(y_val, baseline.predict_proba(X_val)[:,1]))

Baseline AUC: 0.5


In [4]:
# Logistic Regression
model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)
val_preds = model.predict(X_val)
val_proba = model.predict_proba(X_val)[:, 1]

In [5]:
print(classification_report(y_val, val_preds))
print("AUC:", roc_auc_score(y_val, val_proba))

              precision    recall  f1-score   support

           0       0.97      0.42      0.59     13698
           1       0.07      0.74      0.12       773

    accuracy                           0.44     14471
   macro avg       0.52      0.58      0.35     14471
weighted avg       0.92      0.44      0.56     14471

AUC: 0.6006558591475286


## Results Summary

| Model | AUC | Recall (late) | Precision (late) |
|---|---|---|---|
| Baseline (majority class) | 0.50 | 0.00 | - |
| Logistic Regression (v1: order value, payments only) | 0.57 | 0.56 | 0.07 |
| Logistic Regression (v2: + same_state, purchase_month, log_total_price) | 0.60 | 0.74 | 0.07 |

Adding geographic (same_state) and seasonal (purchase_month) features improved 
both AUC and recall meaningfully. Precision remains low — the model over-predicts 
late deliveries. This is a trade-off: high recall protects against missing late 
orders, but the low precision would cause alert fatigue in a real operations team. 
Future work: threshold tuning, or adding more granular location features 
(distance instead of same/different state).